In [ ]:
from pyspark.sql.functions import col, to_timestamp, round as spark_round, date_trunc, unix_timestamp

# 1. Read from our new Bronze tables
orders_bronze = spark.read.table("raw_orders_bronze")
weather_bronze = spark.read.table("raw_weather_bronze")

# 2. Clean Orders: Convert strings to timestamps and calculate delivery duration
orders_cleaned = orders_bronze \
    .withColumn("order_timestamp", to_timestamp(col("order_timestamp"))) \
    .withColumn("delivery_timestamp", to_timestamp(col("delivery_timestamp"))) \
    .withColumn(
        "delivery_duration_minutes", 
        spark_round((unix_timestamp(col("delivery_timestamp")) - unix_timestamp(col("order_timestamp"))) / 60, 2)
    ) \
    .withColumn("join_hour", date_trunc("hour", col("order_timestamp"))) # Round to nearest hour for the join

# 3. Clean Weather: Convert string to timestamp and round to nearest hour
weather_cleaned = weather_bronze \
    .withColumn("weather_time", to_timestamp(col("timestamp"))) \
    .withColumn("join_hour", date_trunc("hour", col("weather_time")))

# 4. JOIN the tables together based on the hour the order was placed
silver_df = orders_cleaned.join(weather_cleaned, on="join_hour", how="left") \
    .drop("join_hour", "weather_time", "timestamp") # Drop extra columns we no longer need

# 5. Save as our Silver Delta Table
silver_df.write.format("delta").mode("overwrite").saveAsTable("orders_weather_silver")

print("Silver Layer Created Successfully! Here is the joined, cleaned data:")
display(silver_df)